# DATA 622: Homework 5
**Author:** Brett Allen

**Date Completed:** TBD (WIP)

## Setup

In [1]:
!python -m pip install -q nltk transformers torch scikit-learn networkx beautifulsoup4 requests

Verify pip module versions

In [2]:
%%bash
for module in nltk transformers torch scikit-learn networkx beautifulsoup4 requests; do echo "====="; echo $module; pip show $module | grep Version; done

=====
nltk
Version: 3.9.2
License: Apache License, Version 2.0
=====
transformers
Version: 4.47.1
=====
torch
Version: 2.5.1
=====
scikit-learn
Version: 1.7.2
=====
networkx
Version: 3.4.2
=====
beautifulsoup4
Version: 4.12.3
=====
requests
Version: 2.32.3


### Imports

In [4]:
import requests
import nltk
import numpy as np
import networkx as nx
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import pipeline

### Configurations

In [5]:
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /home/ballen/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ballen/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Questions
Use the following article:

https://www.usatoday.com/story/news/politics/2025/06/13/pete-hegseth-pentagon-invade-greenland-plan/84188458007/.

In [6]:
url = "https://www.usatoday.com/story/news/politics/2025/06/13/pete-hegseth-pentagon-invade-greenland-plan/84188458007/"

In [7]:
# Use user agent to avoid potential blocking by the website
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
response = requests.get(url, headers=headers)
response.status_code

200

In [8]:
# Parse the HTML content with beautifulsoup
soup = BeautifulSoup(response.text, "html.parser")

# Extract the article text (paragraphs)
paragraphs = soup.find_all("p")
article_text = " ".join([p.get_text() for p in paragraphs])

In [10]:
# Print the first 500 characters of the article text
print(article_text[:500] + "...")

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island. Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies." "It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct? Because I sure as hell...


In [11]:
# Convert article text into sentences
sentences = sent_tokenize(article_text)
print(f"Total sentences: {len(sentences)}")

Total sentences: 16


In [12]:
# Inspect first 5 sentences
print(sentences[:5])

['Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.', 'Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."', '"It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct?', 'Because I sure as hell hope that it is not your testimony," Turner dug in.', '"We look forward to working with Greenland to ensure that it is secured from any potential threats," Hegseth said.']


### 1. Extract and print an extractive summary using the TextRank method.

In [13]:
# # Vectorize sentences using TF-IDF
vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = vectorizer.fit_transform(sentences)

In [14]:
# Create similarity matrix to prepare for TextRank
similarity_matrix = (tfidf_matrix * tfidf_matrix.T).toarray()

In [15]:
# Apply TextRank algorithm using PageRank on the similarity graph
nx_graph = nx.from_numpy_array(similarity_matrix)
scores = nx.pagerank(nx_graph)

In [18]:
scores

{0: 0.07743182158702094,
 1: 0.07220737882730344,
 2: 0.07833435678381329,
 3: 0.05643920408603973,
 4: 0.0648499702801737,
 5: 0.07258043540953774,
 6: 0.05974219806047407,
 7: 0.060321335161830886,
 8: 0.061283536198635524,
 9: 0.06138285157193014,
 10: 0.009900990099009903,
 11: 0.07159118768738873,
 12: 0.057585961548067485,
 13: 0.06461670576290306,
 14: 0.07265646036971052,
 15: 0.05907560656616065}

In [19]:
# Rank sentences based on TextRank scores (sort in descending order so that the highest scored sentences are first)
ranked_sentences = sorted(((scores[i], s) for i, s in enumerate(sentences)), reverse=True)

In [20]:
# Print top 5 ranked sentences as the summary
summary = " ".join([ranked_sentences[i][1] for i in range(5)])

print("\nTextRank Summary:\n")
print(summary)


TextRank Summary:

"It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct? Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island. Greenland belongs to the Greenlanders," Danish Prime Minister Mette Frederiksen said after Vance's visit. President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won't be necessary. Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."


### 2. Extract and print an extractive summary using a frequency-based sentence scoring method.

In [21]:
# Tokenize article text into words and create frequency table for frequency-based sentence scoring method
words = word_tokenize(article_text.lower())

freq_table = {}
for word in words:
    # Only consider alphabetic words (ignore punctuation and numbers)
    if word.isalpha():
        freq_table[word] = freq_table.get(word, 0) + 1

In [22]:
len(freq_table)

153

In [23]:
sentence_scores = {}

# Iterate through sentences and score them based on word frequencies (sum of frequencies of words in the sentence)
for sentence in sentences:
    for word in word_tokenize(sentence.lower()):
        # If the word is in the frequency table, add its frequency to the sentence score
        if word in freq_table:
            sentence_scores[sentence] = sentence_scores.get(sentence, 0) + freq_table[word]

In [24]:
len(sentence_scores)

16

In [ ]:
# Inspect sentence scores
sentence_scores

{'Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.': 90,
 'Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."': 122,
 '"It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct?': 81,
 'Because I sure as hell hope that it is not your testimony," Turner dug in.': 40,
 '"We look forward to working with Greenland to ensure that it is secured from any potential threats," Hegseth said.': 79,
 'President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won\'t be necessary.': 92,
 'He has insisted that acquiring Greenland is necessary for national security, citing growing Chinese and Russian influence in the region.': 70,
 'The

In [26]:
# Rank sentences based on frequency scores (sort in descending order so that the highest scored sentences are first)
ranked_sentences = sorted(sentence_scores, key=sentence_scores.get, reverse=True)

In [27]:
# Create summary by joining the top 5 ranked sentences
summary = " ".join(ranked_sentences[:5])

print("\nFrequency-Based Summary:\n")
print(summary)


Frequency-Based Summary:

During a March visit to Pituffik Space Base, the U.S. base on Greenland, Vice President JD Vance accused Denmark of "failing" to protect the Arctic island while downplaying Trump's threats to take it over by force. In the latest snub to Denmark and other European allies, the Pentagon reportedly plans to move its oversight of the island from U.S. European Command to U.S. Northern Command. Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies." The island is also rich in critical minerals that the U.S. wants to challenge Chinese monopolies in some industries, USA TODAY has reported. President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won't be necessary.


### 3. Generate and print an abstractive summary using a pretrained transformer model (e.g., BART or T5).

### 4. Print a Lead-3 summary (the first three sentences of the article).

### 5. Print a manual compression summary, limiting the result to about 20% of the original sentences.

### 6. Use an LLM to summary the text.